# Learned Embeddings for Categorical Features

Compare three encodings of `country` and `device_type`:
1. One-hot.
2. Learned embeddings (PyTorch `nn.Embedding`).
3. Visualize the learned space — categories with similar fraud risk should cluster.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet').copy()
for c in ['email_risk', 'device_entropy']:
    df[c] = df[c].fillna(df[c].median())

# Integer-encode categoricals for the embedding model.
country_idx = {v: i for i, v in enumerate(sorted(df['country'].unique()))}
device_idx = {v: i for i, v in enumerate(sorted(df['device_type'].unique()))}
df['country_i'] = df['country'].map(country_idx).astype(np.int64)
df['device_i'] = df['device_type'].map(device_idx).astype(np.int64)

numeric_cols = ['account_age_days', 'txn_amount', 'velocity_1h',
                'device_entropy', 'email_risk', 'ip_country_mismatch',
                'noise_1', 'noise_2', 'noise_3']
y = df['is_fraud'].values.astype(np.float32)
X_num = df[numeric_cols].values.astype(np.float32)
X_country = df['country_i'].values
X_device = df['device_i'].values

(X_num_tr, X_num_va, Xc_tr, Xc_va, Xd_tr, Xd_va, y_tr, y_va) = train_test_split(
    X_num, X_country, X_device, y, test_size=0.2, stratify=y, random_state=SEED)

scaler = StandardScaler().fit(X_num_tr)
X_num_tr = scaler.transform(X_num_tr).astype(np.float32)
X_num_va = scaler.transform(X_num_va).astype(np.float32)
print(f"countries: {len(country_idx)}  devices: {len(device_idx)}")

In [ ]:
class FraudWithEmbeddings(nn.Module):
    def __init__(self, n_numeric, n_countries, n_devices, emb_country=4, emb_device=2, hidden=64):
        super().__init__()
        self.emb_country = nn.Embedding(n_countries, emb_country)
        self.emb_device = nn.Embedding(n_devices, emb_device)
        in_dim = n_numeric + emb_country + emb_device
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 1),
        )
    def forward(self, x_num, x_country, x_device):
        e_c = self.emb_country(x_country)
        e_d = self.emb_device(x_device)
        return self.head(torch.cat([x_num, e_c, e_d], dim=1))

model = FraudWithEmbeddings(len(numeric_cols), len(country_idx), len(device_idx))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
pos_weight = torch.tensor((y_tr == 0).sum() / (y_tr == 1).sum(), dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

ds = TensorDataset(
    torch.from_numpy(X_num_tr),
    torch.from_numpy(Xc_tr),
    torch.from_numpy(Xd_tr),
    torch.from_numpy(y_tr).unsqueeze(1),
)
loader = DataLoader(ds, batch_size=256, shuffle=True)

for ep in range(1, 11):
    model.train()
    for xn, xc, xd, yb in loader:
        opt.zero_grad()
        loss_fn(model(xn, xc, xd), yb).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(
            torch.from_numpy(X_num_va),
            torch.from_numpy(Xc_va),
            torch.from_numpy(Xd_va),
        )).numpy().ravel()
    if ep % 2 == 0:
        print(f"ep {ep:02d}  val PR-AUC: {average_precision_score(y_va, probs):.4f}")

In [ ]:
# Inspect the learned country embedding
country_emb = model.emb_country.weight.detach().numpy()
inv_country = {i: v for v, i in country_idx.items()}

# Fraud rate per country, for color coding.
rates = df.groupby('country')['is_fraud'].mean()
colors = [rates[inv_country[i]] for i in range(len(country_idx))]

plt.figure(figsize=(6, 5))
sc = plt.scatter(country_emb[:, 0], country_emb[:, 1], c=colors, cmap='Reds', s=200)
for i, name in inv_country.items():
    plt.annotate(name, (country_emb[i, 0], country_emb[i, 1]),
                 textcoords='offset points', xytext=(6, 6), fontsize=10)
plt.colorbar(sc, label='Empirical fraud rate')
plt.title('Learned country embeddings (first 2 dims)')
plt.xlabel('emb dim 0'); plt.ylabel('emb dim 1')
plt.grid(alpha=0.3); plt.show()

## What you should see

- The model picks up risk structure: high-fraud countries cluster on one side.
- Embeddings can be **exported and reused** by other downstream models (login risk, account-takeover) — that's the whole point of representation learning.

**When to prefer embeddings over one-hot**: high cardinality (zip code, merchant id, device id with thousands of unique values). For our 9 countries, one-hot is fine — but the same code scales to 10k merchants where one-hot would blow up dimensionality.